In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re



In [ ]:
df = pd.read_csv('bbc_news.csv')

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

In [ ]:
lemma = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [ ]:
df.head()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words and len(words)>2]
    words = [lemma.lemmatize(word) for word in words]
    return words

In [ ]:
df['p_disc'] = df['description'].apply(clean_text)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.models import CoherenceModel

In [ ]:
dictionary = gensim.corpora.Dictionary(df['p_disc'])
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in df['p_disc']]
print(corpus)

In [ ]:
def t_and_e(dictionary, corpus, texts, num_topics,alpha,eta):
  lda = LdaModel(
      corpus=corpus,
      id2word=dictionary,
      num_topics=num_topics,
      random_state=42,
      update_every=1,
      chunksize=100,
      passes=10,
      per_word_topics=True,
      alpha = alpha,
      eta = eta
  )
  coherence_model = CoherenceModel(model=lda, texts=df['p_disc'], dictionary=dictionary, coherence='c_v')
  coherence_score = coherence_model.get_coherence()
  return lda, coherence_score

In [ ]:
topics_range = [3, 5, 7, 10]
alpha_vals = ['symmetric', 'asymmetric', 'auto']
eta_vals = ['symmetric', None, 'auto']

results = []
for num in topics_range:
    for alpha in alpha_vals:
        for eta in eta_vals:
            lda_model, coherence = t_and_e(dictionary, corpus, df['p_disc'], num, alpha, eta)
            results.append((num, alpha, eta, coherence))
            print(f"num_topics={num}, alpha={alpha}, eta={eta}, coherence={coherence:.4f}")

In [ ]:
best_prem = max(results, key=lambda x: x[3])
print(f"Best parameters: num_topics={best_prem[0]}, alpha={best_prem[1]}, eta={best_prem[2]}, coherence={best_prem[3]:.4f}")

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

pyLDAvis.enable_notebook()
vis = gensimvis.prepare(lda, corpus, dictionary)
pyLDAvis.display(vis)


In [ ]:
topics = lda.print_topics(num_topics=5, num_words=10)
for idx, topic in topics:
    print(f"Topic {idx}: {topic}")
